# Свип lr: nGPT против GPT (один запуск = одно значение lr)

Общие условия у всех участников: модель 39M (d=384, 6 слоёв, 6 голов), контекст 1024, данные OpenWebText,
одинаковый порядок батчей. Меняются только **lr**, **бюджет** (150M — короткий свип, 775M — полные прогоны) и **сид**.

**Что сделать:**
1. В ячейке настроек поставить своё `LR` из таблицы распределения (и `SEED`, если сказано).
2. Справа: **Accelerator → GPU T4 x2**, **Internet → On**, во **Input** ничего не должно быть.
3. **Save Version → Save & Run All (Commit)**. Браузер можно закрыть. Время: 150M — ~2–3 ч на одно lr, 775M — ~8.5 ч.
4. Когда закончится — **Share** → добавить того, кто собирает общий отчёт (или сделать ноутбук Public).

In [ ]:
# ===== Настройки: менять только эти строки =====
LR = "1.5e-3"     # ← своё значение (или несколько через запятую) из таблицы распределения
BUDGET = "150M"   # короткий свип: 150M; полные прогоны: 775M
SEED = 42         # 42 — первая ночь; 43, 44 — повторы с другим сидом
LRS_GPT = ""      # lr только для GPT (пусто — как LR; "none" — GPT не запускать)
LRS_NGPT = ""     # lr только для nGPT (пусто — как LR; "none" — nGPT не запускать)

SESSION_HOURS = 11.5
TRAIN_TOKENS = "9e8"
REPO = "https://github.com/NordManul/Project_Transformers.git"
PLAN = (f"--n_layer 6 --n_head 6 --d_model 384 --budgets {BUDGET} --lrs {LR} "
        f"--seed {SEED} --data_seed {1234 + SEED - 42}"
        + (f" --lrs_gpt {LRS_GPT}" if LRS_GPT else "") + (f" --lrs_ngpt {LRS_NGPT}" if LRS_NGPT else ""))
NAME = f"sweep_{BUDGET}_lr{LRS_GPT or LR}_{LRS_NGPT or LR}_s{SEED}".replace(",", "+")

import glob, json, os, shutil, subprocess, time
T0 = time.time()
WORK = "/kaggle/working"
CODE = f"{WORK}/Project_Transformers"
DATA = f"{WORK}/data/owt"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
os.chdir(CODE)
print("код:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
import torch
print("torch", torch.__version__, "| GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

OUT = f"{WORK}/runs/{NAME}"
print("PLAN:", PLAN)
print("OUT:", OUT)

## 1. Данные (900M токенов OpenWebText, ~14 мин; если есть во входе — копируются)

In [ ]:
found = sorted(glob.glob("/kaggle/input/**/data/owt/train.bin", recursive=True))
if os.path.exists(f"{DATA}/train.bin"):
    print("Данные уже на месте")
elif found and json.load(open(os.path.join(os.path.dirname(found[0]), "meta.json")))["train_tokens"] >= 9e8:
    print("Копирую данные из входа:", os.path.dirname(found[0]))
    shutil.copytree(os.path.dirname(found[0]), DATA, dirs_exist_ok=True)
else:
    !python tokenize_dataset.py --dataset openwebtext --train_tokens {TRAIN_TOKENS} --val_tokens 1e7 --out {DATA}
print(open(f"{DATA}/meta.json").read())

## 2. Обучение (GPT и nGPT одновременно, каждая на своей T4)

In [ ]:
HOURS = f"{SESSION_HOURS - (time.time() - T0) / 3600 - 0.3:.2f}"
print("Часов на обучение:", HOURS)
!python ngpt_vs_gpt.py run --data {DATA} --out {OUT} {PLAN} --hours {HOURS}

## 3. Отчёт этого запуска

In [ ]:
!python ngpt_vs_gpt.py report --out {OUT} > /dev/null
!python make_pdf_report.py {OUT}/report --out /kaggle/working/{NAME}.pdf
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/report/report.md", encoding="utf-8").read()))